<a href="https://colab.research.google.com/github/EricStimpsonWSU/CHiMaD-PFC-Demo/blob/main/PFC_Model_step2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simple PFC

This simple PFC Simulation implements the following model:

$$
\partial_t \psi = \Gamma \nabla^2 \mu
$$

with

$$
\mu = B(\nabla^2 + q_0^2)^2 \psi + r \psi + g \psi^2 + v_0 \psi^3
$$

where $q_0$ sets the real-space length scale $a_0 = 4 \pi / \sqrt{3}$ for the hexagonal (triangle) lattice:

$\psi(\mathbf{r})$ for hexagonal lattice:
```
 o o o o
o o o o
 o o o o
o o o o
```

This algorithm in this implementation calculates field derivatives in Fourier space and products (powers) in real space with exponential timestepping.  The algorithm is highly parallelizable, well-suited to multicore CPUs or GPUs.  Finally, the implementation utilizes a data-layer abstraction to ease switching between CPU / GPU runtimes.

##### Time stepping algorithm:



## PFC Simulation Class

In [ ]:
# @markdown ##### CPU / GPU agnostic arrays numpy / cupy as xp
try:
    import cupy as xp
    print("Using CuPy (GPU) backend")

    def to_host_array(arr):
        """Helper function to get array data from device to host."""
        return arr.get()

except ImportError:
    import numpy as xp
    print("Using NumPy (CPU) backend")

    def to_host_array(arr):
        """Helper function to get array data from device to host."""
        return arr # NumPy arrays are already on the CPU

In [ ]:
from re import S
class sPFC:
  """Stateful sPFC simulation object

  This class maintains the state of a simple phase-field crystal (PFC)
  model and includes methods for creating the simulation box,
  setting the initial state, definining the simulation parameters,
  and running the simulation.  Datalayer is agnostic toward computation
  on CPU / GPU.  Arrays (Buffers) are written in-place.

  Key Attributes:
  - Nx (int): Number of grid points in the x-direction.
  - Ny (int): Number of grid points in the y-direction.
  - dx (float): Grid spacing in the x-direction.
  - dy (float): Grid spacing in the y-direction.
  - dt (float): Timestep.

  Key Buffers:
  - psi (numpy.ndarray): Real-space density field.
  - psi_hat (numpy.ndarray): Fourier-space density field.

  Initialization:
  - InitGeometry(Nx, Ny, dx, dy): Initializes the simulation box
  - SetInitialState(psi0, sigma): Sets the initial state of the simulation
  - SetModelParams(B, r, g, v0, dt): Sets PFC model parameters
  - Run(steps): Runs the simulation for the specified number of steps
  """

  # Constructor
  def __init__(self) -> None:
    return None

  # Printing
  def __repr__(self) -> str:
    msg = 'Simple PFC Simulation'
    if hasattr(self, 'Nx'):
      # Get k-space resolution values
      dkx_val = self.KX[0, 1]
      dky_val = self.KY[1, 0]

      msg += (
          f"\n"
          f"  Simulation box: {self.Nx}x{self.Ny} (Nx x Ny)\n"
          f"  Real-space dimensions: {self.Lx:.4f} x {self.Ly:.4f} (Lx x Ly)\n"
          f"  Element size: {self.dx:.4f} x {self.dy:.4f} (dx x dy)\n"
          f"  K-space resolution: {dkx_val:.4f} x {dky_val:.4f} (dkx x dky)"
      )
    if hasattr(self, 'psi'):
      msg += (
          f"\n"
          f"  Initial density: {self.psi0:.4f}"
          f" +/- {self.psi_sigma:.4f}"
      )
    if hasattr(self, 't') and self.t > 0:
      msg += (
          f"\n"
          f"  Current density: {self.psi.mean():.4f}"
          f" +/- {self.psi.std():.4f}"
      )
    return msg

  def InitGeometry(self, Nx, Ny, dx, dy) -> None:
    """ Create simulation box...

    Args:
    - Nx (int): Number of grid points in the x-direction.
    - Ny (int): Number of grid points in the y-direction.
    - dx (float): Grid spacing in the x-direction.
    - dy (float): Grid spacing in the y-direction

    Description:
    This method initializes the simulation grid and calculates the real-space
    and k-space dimensions.

    Returns:
    None
    """
    self.Nx = Nx
    self.Ny = Ny
    self.dx = dx
    self.dy = dy
    self.Lx = Nx * dx
    self.Ly = Ny * dy

    self.x = xp.linspace(0, self.Lx, self.Nx, endpoint=False)
    self.y = xp.linspace(0, self.Ly, self.Ny, endpoint=False)

    self.X, self.Y = xp.meshgrid(self.x, self.y)

    kx = xp.fft.fftfreq(self.Nx, self.dx) * 2 * xp.pi
    ky = xp.fft.fftfreq(self.Ny, self.dy) * 2 * xp.pi

    self.KX, self.KY = xp.meshgrid(kx, ky)

    return None

  def SetInitialState(self, psi0, sigma) -> None:
    """ Set initial simulation state using a Gaussian distribution.

    Args:
    - psi0 (numpy.ndarray): Average density.
    - sigma (numpy.ndarray): Standard deviation of the Gaussian distribution.

    Description:
    This method sets the initial state of the simulation using
    specified average density and a Gaussian distribution.  The
    average density is exact [optionally, low-pass filter].

    Returns:
    None
    """

    # create real and complex density fields for real and Fourier space
    size = (self.Ny, self.Nx)
    self.psi = xp.zeros(size, dtype = xp.float64)
    self.psi_hat = xp.zeros(size, dtype = xp.complex128)

    # initialize density with normal distribution
    self.psi[...] = xp.random.normal(loc=0, scale=sigma, size=(self.Ny, self.Nx))

    # Normalize to psi0 in k-space
    self.psi_hat[...] = xp.fft.fftn(self.psi)
    self.psi_hat[0,0] = psi0 * self.Nx * self.Ny

    # optional low-pass filter part 1:
    self.psi_hat[(self.KX**2 + self.KY**2) > (self.KX[0,16]**2 + self.KY[16,0]**2)] = 0

    self.psi[...] = xp.real(xp.fft.ifftn(self.psi_hat))

    # optional low-pass filter part 2 [sets or corrects distribution sigma]:
    self.psi[...] = (self.psi - self.psi.mean()) / self.psi.std() * sigma + self.psi.mean()

    # store initial state statistics
    self.psi0 = psi0
    self.psi_sigma = self.psi.std()
    return None

  def SetModelParams(self, B, r, g, v0, dt) -> None:
    """ Set simulation parameters.

    Args:
    - B: Rigidity.
    - r: Temperature.
    - g: Cubic term.
    - v0: Quartic term.
    - dt: Timestep.

    Description:
    This method sets the PFC model parameters and calculates kernels used
    in timestepping.  Buffers are allocated for intermediate calculations.

    Returns:
    None
    """
    self.B = B
    self.r = r
    self.g = g
    self.v0 = v0
    self.dt = dt    # Δt

    # create real and complex buffers for timestep computation
    size = (self.Ny, self.Nx)
    self.psi2_hat = xp.zeros(size, dtype = xp.complex128)
    self.psi3_hat = xp.zeros(size, dtype = xp.complex128)
    self.nonlin_mu_hat = xp.zeros(size, dtype = xp.complex128)
    self.psi_t_p_dt_hat = xp.zeros(size, dtype = xp.complex128)
    self.kernel_d2_dlap = xp.zeros(size, dtype = xp.float64)
    self.kernel_d4_dlap2 = xp.zeros(size, dtype = xp.float64)
    self.kernel_d6_dlap3 = xp.zeros(size, dtype = xp.float64)
    self.kernel_lin_dpsidt = xp.zeros(size, dtype = xp.float64)
    self.kernel_linexp_dpsidt = xp.zeros(size, dtype = xp.float64)
    self.kernel_nonlinexp_dpsidt = xp.zeros(size, dtype = xp.float64)

    # define kernels in Fourier space
    self.kernel_d2_dlap[...] = -(self.KX * self.KX + self.KY * self.KY)         # ∇^2
    self.kernel_d4_dlap2[...] = self.kernel_d2_dlap * self.kernel_d2_dlap       # ∇^4
    self.kernel_d6_dlap3[...] = self.kernel_d4_dlap2 * self.kernel_d2_dlap      # ∇^6

    # define linear timestep kernel
    self.kernel_lin_dpsidt[...] = (
        (self.r + self.B) * self.kernel_d2_dlap +
        2 * self.B * self.kernel_d4_dlap2 +
        self.B * self.kernel_d6_dlap3
      )                                                                         # α

    # define linear and nonlinear exponential timestep kernel
    self.kernel_linexp_dpsidt[...] = xp.exp(self.kernel_lin_dpsidt * self.dt)

    self.kernel_nonlinexp_dpsidt[...] = xp.ones_like(self.kernel_lin_dpsidt) * self.dt
    self.kernel_nonlinexp_dpsidt[self.kernel_lin_dpsidt != 0] = (
        (self.kernel_linexp_dpsidt[self.kernel_lin_dpsidt != 0] - 1) / self.kernel_lin_dpsidt[self.kernel_lin_dpsidt != 0]
    )
    self.t = 0. * self.dt

    return None

  def Run(self, steps) -> None:
    """ Run simulation...

    Args:
    - steps (int): Number of timesteps to run.

    Description:
    This method runs the simulation for the specified number of steps
    using exponential timestepping.

    Returns:
    None
    """

    for step in range(steps):
      # build nonlinear term in Fourier space
      self.psi2_hat[...] = xp.fft.fftn(self.psi * self.psi)
      self.psi3_hat[...] = xp.fft.fftn(self.psi * self.psi * self.psi)
      self.nonlin_mu_hat[...] = (
          self.g * self.psi2_hat +
          self.v0 * self.psi3_hat)

      # calculate ψ[t + Δt](k) in Fourier space
      self.psi_hat[...] = xp.fft.fftn(self.psi)
      self.psi_t_p_dt_hat[...] = (
          self.kernel_linexp_dpsidt * self.psi_hat +
          self.kernel_d2_dlap * self.kernel_nonlinexp_dpsidt * self.nonlin_mu_hat
      )

      # convert to density to realspace
      self.psi[...] = xp.real(xp.fft.ifftn(self.psi_t_p_dt_hat))

      # update time
      self.t += self.dt

    return None

## PFC Simulation Test

### Create simulation

In [ ]:
# Create a PFC sim
sim = sPFC()
print(sim)

In [ ]:
# Initialize the simulation box
sim.InitGeometry(2**10, 2**10, 0.5, 0.5)
print(sim)

In [ ]:
# Set initial state
sim.SetInitialState(-0.4, 0.5)
print(sim)

In [ ]:
# Set model parameters for timestepping
sim.SetModelParams(
    B = 1,
    r = -0.5,
    g = 0,
    v0 = 1,
    dt = 1e-1)
print(sim)

### View initial state

In [ ]:
# sim.psi[...] = xp.ones((sim.Ny, sim.Nx), dtype = xp.float64) * -0.4
# sim.psi[int(sim.Ny/2), int(sim.Nx/2)] += .1

In [ ]:
import plotly.graph_objects as go

x = to_host_array(sim.x) # use 1D arrays for heatmap coords
y = to_host_array(sim.y)
Z = to_host_array(sim.psi)
fig = go.Figure(data=[go.Heatmap(
    x = x,
    y = y,
    z = Z,
    colorscale='Greys_r'
)])
fig.update_layout(
    title = 'Initial State',
    xaxis_title = 'x',
    yaxis_title = 'y',
    xaxis=dict(scaleanchor="y", scaleratio=1), # Ensure same scaling for x and y
    yaxis=dict(scaleanchor="x", scaleratio=1),
    width=600,  # Set a fixed width for consistent aspect ratio
    height=600, # Set a fixed height for consistent aspect ratio
    autosize=False
)
fig.show()

### Run Simulation

In [ ]:
# Run the simulation for 1000 steps
sim.Run(1000)
print(sim)

### View Final State

In [ ]:
import plotly.graph_objects as go

x = to_host_array(sim.x) # use 1D arrays for heatmap coords
y = to_host_array(sim.y)
Z = to_host_array(sim.psi)
fig = go.Figure(data=[go.Heatmap(
    x = x,
    y = y,
    z = Z,
    colorscale='Greys_r'
)])
fig.update_layout(
    title = 'Final State',
    xaxis_title = 'x',
    yaxis_title = 'y',
    xaxis=dict(scaleanchor="y", scaleratio=1), # Ensure same scaling for x and y
    yaxis=dict(scaleanchor="x", scaleratio=1),
    width=600,  # Set a fixed width for consistent aspect ratio
    height=600, # Set a fixed height for consistent aspect ratio
    autosize=False
)
fig.show()